# Topic Modeling — LDA and BERTopic

**Author:** Shivani Bokka  
**Dataset:** 20 Newsgroups (sklearn built-in — standard NLP benchmark)  
**Goal:** Discover hidden topics in text corpora without any labels

---

## What Is This Notebook About?

This notebook is a **complete walkthrough of topic modeling** — the art of finding hidden themes in large collections of text, entirely without human labels. Each section explains the core idea in plain language before diving into the code.

Whether you're new to NLP or looking for a clean reference on LDA versus BERTopic, this notebook covers both approaches end-to-end.

---

## What Is Topic Modeling?

> **Given a collection of documents, topic modeling discovers the hidden themes automatically — without any labels.**

Imagine you have 10,000 customer reviews and no idea what people are talking about. Topic modeling reads all of them and says: "It looks like people are discussing *product quality*, *shipping speed*, and *customer service*." You never had to label a single document.

**Real-world uses:**
- Customer feedback analysis — what are people actually complaining about?
- News categorization — grouping articles by theme without human editors
- Research paper clustering — finding research trends in scientific literature
- Social media monitoring — tracking what topics are trending and why

---

## Models Covered in This Notebook

| # | Model | Key Idea |
|---|-------|----------|
| 1 | LDA (Latent Dirichlet Allocation) | Statistical bag-of-words approach; each doc is a mixture of topics |
| 2 | BERTopic | Neural embedding approach; uses BERT + UMAP + HDBSCAN |

---

---

## Section 1 — Imports and Setup

Before anything else, we import all the tools we'll need throughout this notebook.

- **numpy / pandas** — for data manipulation
- **matplotlib / seaborn** — for visualization
- **sklearn** — for LDA, vectorizers, and the 20 Newsgroups dataset
- **nltk** — for text preprocessing (stopwords, lemmatization)
- **gensim** — for coherence score computation
- **pyLDAvis** — for interactive LDA visualization
- **bertopic** — for the BERTopic neural topic model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import warnings
import re
import string
warnings.filterwarnings('ignore')

# Sklearn — dataset and models
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# NLTK — text preprocessing
import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Gensim — coherence scoring
import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel

# Display settings
pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
matplotlib.rcParams['font.size'] = 11

print("All libraries imported successfully!")
print(f"Gensim version: {gensim.__version__}")

---

## Section 2 — What Is Topic Modeling?

### Two Fundamentally Different Approaches

| Approach | LDA | BERTopic |
|----------|-----|----------|
| Text representation | Bag-of-words (word counts) | Dense neural embeddings (BERT) |
| Statistical model | Dirichlet distribution | UMAP + HDBSCAN clustering |
| Word order | Ignored | Partially captured |
| Number of topics | You must specify upfront | Auto-discovered |
| Handles short texts | Poorly | Well |
| Speed | Fast | Slow (GPU helps a lot) |
| Best for | Long documents, many docs, known topic count | Short texts, semantic understanding, unknown topic count |

---

### Load the 20 Newsgroups Dataset

**20 Newsgroups** is a classic NLP benchmark dataset. It contains approximately 18,000 newsgroup posts organized into 20 topic categories. We'll use a manageable subset of **4 categories**:

| Category | What It Contains |
|----------|------------------|
| `rec.autos` | Car discussions, reviews, buying advice |
| `sci.med` | Medical science, health questions, treatments |
| `comp.graphics` | Computer graphics, software, rendering |
| `talk.politics.guns` | Gun rights, policy debates, legislation |

These 4 categories were chosen because they are **semantically distinct** — a good topic model should clearly separate them.

In [ ]:
# Load 4 categories from 20 Newsgroups
categories = ['rec.autos', 'sci.med', 'comp.graphics', 'talk.politics.guns']

newsgroups = fetch_20newsgroups(
    subset='all',
    categories=categories,
    remove=('headers', 'footers', 'quotes'),  # Remove metadata — cleaner text
    random_state=42
)

docs = newsgroups.data
true_labels = newsgroups.target
label_names = newsgroups.target_names

print(f"Total documents loaded: {len(docs)}")
print(f"Categories: {label_names}")
print(f"\nDocuments per category:")
for i, name in enumerate(label_names):
    count = sum(1 for t in true_labels if t == i)
    print(f"  {name}: {count} documents")

print(f"\n--- Sample Document (raw, unprocessed) ---")
print(docs[0][:800])

---

## Section 3 — Text Preprocessing Pipeline

Raw text is messy. Before feeding documents to a topic model, we need to clean and normalize the text. Topic models — especially LDA — are very sensitive to the quality of preprocessing.

**Bad preprocessing → garbage topics. Always verify with a before/after comparison.**

### The 5-Step Preprocessing Pipeline

| Step | What It Does | Why It Matters |
|------|-------------|----------------|
| 1. Lowercase | `"Car"` → `"car"` | "Car" and "car" should be the same word |
| 2. Remove punctuation & numbers | `"3.5L engine!"` → `"L engine"` | Numbers and symbols add noise |
| 3. Tokenize | `"the car"` → `["the", "car"]` | Split into individual words |
| 4. Remove stopwords | Remove "the", "is", "a", etc. | Common words appear everywhere; they don't help distinguish topics |
| 5. Lemmatize | `"running"` → `"run"`, `"cars"` → `"car"` | Normalize word forms so we count them together |

In [ ]:
# ── PREPROCESSING PIPELINE ──────────────────────────────────────────────────

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    """Clean and normalize a single document."""
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    # Step 4: Remove stopwords and very short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # Step 5: Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Show before/after for the first document
raw_sample = docs[0]
clean_sample = preprocess(raw_sample)

print("=" * 60)
print("BEFORE PREPROCESSING (first 500 chars):")
print("=" * 60)
print(raw_sample[:500])
print()
print("=" * 60)
print("AFTER PREPROCESSING (first 500 chars):")
print("=" * 60)
print(clean_sample[:500])

### How to Read This: Text Preprocessing

Compare the two outputs side by side:

- **Before:** Raw newsgroup text. Noisy, contains headers, punctuation, numbers, very common English words ("the", "is", "of"), and different forms of the same word ("running", "runs", "ran").
- **After:** Clean, normalized tokens. What's left are the **meaningful words** that carry the actual semantic content of the document.

**Why this matters for topic modeling:**
- LDA treats each unique word as a separate vocabulary item. Without preprocessing, "Car", "car", "cars", and "car's" would all be different words — wasting probability mass.
- Stopwords like "the" and "a" appear in **every** document and would dominate every topic if not removed.
- After preprocessing, the topic model has a much cleaner signal to work with.

In [ ]:
# Preprocess all documents
print("Preprocessing all documents... (this may take a minute)")
processed_docs = [preprocess(doc) for doc in docs]
print(f"Done. Processed {len(processed_docs)} documents.")

# Filter out empty documents
valid_mask = [len(d.strip()) > 0 for d in processed_docs]
processed_docs = [d for d, v in zip(processed_docs, valid_mask) if v]
true_labels_filtered = [l for l, v in zip(true_labels, valid_mask) if v]
raw_docs_filtered = [d for d, v in zip(docs, valid_mask) if v]

print(f"Valid documents after filtering: {len(processed_docs)}")

In [ ]:
# ── BUILD COUNTVECTORIZER ────────────────────────────────────────────────────

count_vectorizer = CountVectorizer(
    min_df=5,         # Word must appear in at least 5 documents
    max_df=0.95,      # Word must not appear in more than 95% of documents
    stop_words='english',
    max_features=5000
)

dtm_counts = count_vectorizer.fit_transform(processed_docs)
vocab = count_vectorizer.get_feature_names_out()

print(f"Vocabulary size: {len(vocab)} words")
print(f"Document-term matrix shape: {dtm_counts.shape}")
print(f"  (rows = documents, columns = words)")

# Also build TF-IDF vectorizer (useful for comparison)
tfidf_vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.95,
    stop_words='english',
    max_features=5000
)
dtm_tfidf = tfidf_vectorizer.fit_transform(processed_docs)
print(f"\nTF-IDF matrix shape: {dtm_tfidf.shape}")

In [ ]:
# ── WORD FREQUENCY DISTRIBUTION ─────────────────────────────────────────────

# Sum word counts across all documents
word_counts = np.array(dtm_counts.sum(axis=0)).flatten()
word_freq_df = pd.DataFrame({'word': vocab, 'count': word_counts})
word_freq_df = word_freq_df.sort_values('count', ascending=False).head(30)

plt.figure(figsize=(12, 5))
sns.barplot(data=word_freq_df, x='word', y='count', palette='Blues_r')
plt.title('Top 30 Most Frequent Words Across All Documents', fontsize=13, fontweight='bold')
plt.xlabel('Word')
plt.ylabel('Total Count Across Corpus')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### How to Read This Chart: Word Frequency Distribution

This bar chart shows the **30 most common words** across all documents in the corpus (after stopword removal and lemmatization).

- **Each bar** represents one word. **Height = total number of times it appears** across all documents.
- Words on the **left** are the most common overall. Words on the **right** are less frequent (but still in the top 30).

**What to look for:**
- Very high-frequency words like "said", "know", "think" appear in **multiple topics** — they're general discussion words. These won't help distinguish topics.
- Words that are clearly category-specific ("car", "gun", "medical", "image") are gold — they're the backbone of a good topic.
- The `max_df=0.95` parameter in CountVectorizer already filtered out words that appeared in more than 95% of documents. This catches words like "people" that are still not informative.

> **Tip:** If you see a lot of clearly non-content words still in the top 30, consider adding them to a custom stopword list before re-running the vectorizer.

---

## Section 4 — LDA (Latent Dirichlet Allocation)

### How LDA Works

> **LDA assumes each document is a mixture of topics, and each topic is a mixture of words.**

Here's the intuition in plain English:

Imagine a person writing a newsgroup post about their car breaking down at a hospital. Their post is partly about **automobiles** (car, engine, repair) and partly about **medicine** (doctor, hospital, treatment). LDA models this as:

```
Document: "My car broke down near the hospital..."
    ↓
Topic Distribution: [60% autos, 30% medicine, 5% graphics, 5% politics]
    ↓
Each topic → probability distribution over words:
  Topic "autos"   : car(0.08), engine(0.06), drive(0.05), oil(0.04), ...
  Topic "medicine": doctor(0.07), patient(0.06), drug(0.05), health(0.04), ...
```

**The LDA generative story (simplified):**
1. For each document, draw a topic mixture from a Dirichlet distribution
2. For each word slot in the document, pick a topic from that mixture
3. Then pick a word from that topic's word distribution

Training reverses this: given the observed words, infer the topic distributions.

### Key Parameters

| Parameter | What It Controls | Typical Values |
|-----------|-----------------|----------------|
| `n_components` | Number of topics | 5–50 depending on corpus size |
| `alpha` (doc-topic) | How spread topics are per document; lower = more focused | Auto |
| `eta` (topic-word) | How spread words are per topic; lower = more focused | Auto |

In [ ]:
# ── TRAIN LDA WITH 4 TOPICS ──────────────────────────────────────────────────
# We use 4 because we know there are 4 categories in our dataset.
# In real projects, you'd use the coherence score (Section 5) to choose this.

lda_model = LatentDirichletAllocation(
    n_components=4,
    max_iter=30,
    learning_method='online',
    random_state=42,
    n_jobs=-1
)

print("Training LDA model...")
doc_topic_matrix = lda_model.fit_transform(dtm_counts)
print("Done!")
print(f"\nDocument-topic matrix shape: {doc_topic_matrix.shape}")
print(f"  Each row = one document, each column = probability for that topic")
print(f"\nExample — first document's topic distribution:")
print(f"  {dict(zip([f'Topic {i}' for i in range(4)], doc_topic_matrix[0].round(3)))}")

In [ ]:
# ── PRINT TOP WORDS PER TOPIC ────────────────────────────────────────────────

def print_top_words(model, feature_names, n_top_words=10):
    """Print the top words for each LDA topic."""
    print("=" * 60)
    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[-n_top_words:][::-1]
        top_words = [feature_names[i] for i in top_indices]
        print(f"\nTopic {topic_idx}:")
        print(f"  Top words: {', '.join(top_words)}")
        # Rough interpretation based on words
        print(f"  → What topic is this? Look at the words above and give it a human-readable name.")
    print("\n" + "=" * 60)

print_top_words(lda_model, vocab, n_top_words=10)

### How to Read This Output: LDA Topics

LDA has discovered 4 topics. For each topic, we show the **10 words with the highest probability** in that topic's word distribution.

**How to interpret:**
- **Each topic is a probability distribution over all words in the vocabulary.** The top words are the ones with the highest probability — the words most likely to appear when this topic is active.
- **Your job is to name the topic** by reading the top words and finding the common theme. For example:
  - Words like `car, engine, drive, oil, transmission` → **Automobiles**
  - Words like `doctor, patient, drug, disease, hospital` → **Medicine**
  - Words like `image, file, color, format, software` → **Computer Graphics**
  - Words like `gun, weapon, law, government, right` → **Gun Politics**

**Important caveats:**
- Topics are **not labeled** by LDA — you assign the labels manually based on the words.
- A "messy" topic with words from multiple domains means LDA is struggling to separate those themes. Consider more preprocessing or adjusting `n_components`.
- The **order of topics is arbitrary** — Topic 0 is not necessarily more important than Topic 3.

---

## Section 5 — Choosing Number of Topics (Coherence Score Curve)

### The Central Question: How Many Topics?

LDA requires you to specify the number of topics (`n_components`) **before training**. Choose too few and topics are vague blobs. Choose too many and topics fragment into meaningless micro-clusters.

**Two metrics help you choose:**

| Metric | What It Measures | Direction |
|--------|-----------------|----------|
| **Coherence Score (c_v)** | How often the top words of a topic co-occur in documents. Higher = more interpretable topics | Higher is better |
| **Perplexity** | How surprised the model is by held-out documents. Lower = model fits the data better statistically | Lower is better |

> **Key insight:** Lower perplexity does NOT always mean more interpretable topics. You can have mathematically perfect topics that no human can understand. Always use coherence as your primary metric — it correlates better with human judgment.

In [ ]:
# ── COHERENCE AND PERPLEXITY ACROSS TOPIC COUNTS ────────────────────────────

# Prepare gensim corpus for coherence scoring
tokenized_docs = [doc.split() for doc in processed_docs]
dictionary = corpora.Dictionary(tokenized_docs)
dictionary.filter_extremes(no_below=5, no_above=0.95)
corpus = [dictionary.doc2bow(doc) for doc in tokenized_docs]

topic_range = [2, 4, 6, 8, 10, 12, 15]
coherence_scores = []
perplexity_scores = []

print("Training LDA models for different topic counts...")
print("(This will take a few minutes)")

for n in topic_range:
    # Train sklearn LDA for perplexity
    lda_temp = LatentDirichletAllocation(
        n_components=n,
        max_iter=20,
        learning_method='online',
        random_state=42
    )
    lda_temp.fit(dtm_counts)
    perp = lda_temp.perplexity(dtm_counts)
    perplexity_scores.append(perp)

    # Compute coherence using gensim
    # Extract top words per topic from sklearn LDA
    topics_words = []
    for topic in lda_temp.components_:
        top_word_ids = topic.argsort()[-10:][::-1]
        top_words = [vocab[i] for i in top_word_ids if vocab[i] in dictionary.token2id]
        topics_words.append(top_words)

    coherence_model = CoherenceModel(
        topics=topics_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence_scores.append(coherence_model.get_coherence())
    print(f"  n_topics={n:2d} | Coherence={coherence_scores[-1]:.4f} | Perplexity={perp:.1f}")

print("\nDone!")

In [ ]:
# ── PLOT COHERENCE AND PERPLEXITY ────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Coherence score ---
axes[0].plot(topic_range, coherence_scores, marker='o', color='steelblue', linewidth=2, markersize=8)
best_coherence_n = topic_range[np.argmax(coherence_scores)]
axes[0].axvline(best_coherence_n, color='red', linestyle='--', label=f'Best: n={best_coherence_n}')
axes[0].set_title('Coherence Score vs Number of Topics', fontweight='bold')
axes[0].set_xlabel('Number of Topics')
axes[0].set_ylabel('Coherence Score (c_v)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Right: Perplexity ---
axes[1].plot(topic_range, perplexity_scores, marker='s', color='coral', linewidth=2, markersize=8)
axes[1].set_title('Perplexity vs Number of Topics', fontweight='bold')
axes[1].set_xlabel('Number of Topics')
axes[1].set_ylabel('Perplexity (lower = better statistical fit)')
axes[1].grid(True, alpha=0.3)

plt.suptitle('LDA Model Selection: Coherence and Perplexity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best coherence score at n_topics = {best_coherence_n}: {max(coherence_scores):.4f}")

### How to Read This Chart: Coherence vs Number of Topics

**Left panel — Coherence Score:**
- **Y-axis (higher = better):** The coherence score (c_v) measures how often the top words of each topic appear together in the same documents. Coherent topics have words that naturally co-occur.
- **X-axis:** Number of topics tested.
- **What to look for:** The **peak** or **elbow** in the coherence curve. At the peak, topics are maximally interpretable. After the peak, adding more topics just fragments the existing clean topics.
- **The red dashed line** marks the number of topics with the highest coherence score.

**Right panel — Perplexity:**
- **Y-axis (lower = better):** Perplexity measures how well the model predicts held-out documents. Think of it as the model's surprise — low perplexity means the model is not surprised by new documents.
- **Perplexity typically decreases monotonically** as you add more topics — the model always fits the data better with more parameters.
- **Do NOT use perplexity alone to choose your topic count.** A model with 50 topics might have lower perplexity than one with 4 topics but be impossible to interpret.

> **Decision rule:** Find where coherence peaks (or elbows). Cross-check with perplexity. If coherence peaks at n=4 but perplexity is much lower at n=10, you might try n=6 as a compromise.

---

## Section 6 — pyLDAvis Interactive Visualization

pyLDAvis creates a powerful interactive visualization that lets you explore LDA topics visually. It shows two panels:

- **Left panel — Topic bubbles:** Each circle is one topic. Circle size = how prevalent that topic is across all documents (bigger = more common). Distance between circles = how similar the topics are (far apart = very different). Overlapping circles = topics share vocabulary.
- **Right panel — Word bars:** When you select a topic, this shows the top words. Blue bars = how common the word is overall. Red bars = how much this word is "lifted" by this topic (topic-specific importance).

### The Lambda (λ) Slider
pyLDAvis has a slider (λ) that goes from 0 to 1:
- **λ = 1:** Ranks words purely by frequency within the topic — you see the most common topic words
- **λ = 0:** Ranks words purely by their "lift" (how specific they are to this topic vs other topics) — you see the most distinctive words
- **λ ≈ 0.6:** Research suggests this gives the most interpretable word ranking for humans

In [ ]:
# ── INSTALL AND IMPORT PYLLDAVIS ─────────────────────────────────────────────
# If not installed, uncomment the next line:
# !pip install pyLDAvis

try:
    import pyLDAvis
    import pyLDAvis.lda_model as lda_vis  # sklearn interface
    print("pyLDAvis imported successfully.")
    PYLLDAVIS_AVAILABLE = True
except ImportError:
    print("pyLDAvis is not installed. Run: pip install pyLDAvis")
    print("Skipping interactive visualization.")
    PYLLDAVIS_AVAILABLE = False

In [ ]:
# ── PREPARE AND SAVE PYLLDAVIS VISUALIZATION ─────────────────────────────────

if PYLLDAVIS_AVAILABLE:
    print("Preparing pyLDAvis visualization...")
    vis_data = lda_vis.prepare(
        lda_model,
        dtm_counts,
        count_vectorizer,
        mds='mmds',
        sort_topics=False
    )
    output_path = 'lda_visualization.html'
    pyLDAvis.save_html(vis_data, output_path)
    print(f"\nVisualization saved to: {output_path}")
    print("Open this HTML file in your browser to interact with the topic map.")
    print("\nThe visualization shows:")
    print("  - Left: Topic bubbles (size = prevalence, distance = similarity)")
    print("  - Right: Word bars for selected topic (blue = overall freq, red = topic lift)")
else:
    print("pyLDAvis not available — skipping HTML export.")
    print("Install with: pip install pyLDAvis")

### How to Read pyLDAvis

The pyLDAvis visualization is saved as `lda_visualization.html`. Open it in any browser. Here is how to use it:

**Left panel — The Topic Map:**
- Each numbered circle is one LDA topic.
- **Circle size:** Proportional to the topic's prevalence across all documents. A large circle means that topic appears in many documents.
- **Distance between circles:** Topics that are far apart share few words and are semantically different. Topics that overlap share many words and may be too similar — a sign you have too many topics.
- **Ideal picture:** You want 4 clearly separated, evenly-sized circles with no overlaps. Overlaps = topic redundancy.

**Right panel — Word Relevance:**
- Click on any topic bubble to see its top words.
- **Blue bar length:** How common the word is in the entire corpus.
- **Red bar length:** How strongly the word is associated with *this* topic (relative to all other topics).
- Words where red >> blue are the **most distinctive words** for that topic — the ones that set it apart.

**The λ slider:**
- Moving λ toward 0 highlights the most *distinctive* words (even if rare).
- Moving λ toward 1 highlights the most *frequent* words within the topic.
- Try λ = 0.6 as a starting point — research shows this is the most interpretable setting for humans.

---

## Section 7 — Document-Topic Assignment

After training LDA, every document gets a **topic distribution** — a set of probabilities showing how much of each topic is present in that document.

We can use this to:
1. Find which documents best represent each topic (highest topic probability)
2. Understand how "pure" or "mixed" each document is
3. Assign each document to its dominant topic

In [ ]:
# ── TOP DOCUMENTS PER TOPIC ──────────────────────────────────────────────────

# Assign each document to its dominant topic
dominant_topics = np.argmax(doc_topic_matrix, axis=1)

# Give our topics human-readable names based on what we saw in Section 4
topic_names = ['Topic 0', 'Topic 1', 'Topic 2', 'Topic 3']

print("TOP 3 DOCUMENTS PER TOPIC")
print("=" * 70)

for topic_id in range(4):
    # Get documents where this is the dominant topic, sorted by probability
    topic_probs = doc_topic_matrix[:, topic_id]
    top_doc_indices = topic_probs.argsort()[-3:][::-1]

    print(f"\n{'=' * 70}")
    print(f"TOPIC {topic_id} — Top Words: {', '.join([vocab[i] for i in lda_model.components_[topic_id].argsort()[-8:][::-1]])}")
    print(f"{'=' * 70}")

    for rank, idx in enumerate(top_doc_indices, 1):
        topic_dist = doc_topic_matrix[idx]
        true_cat = label_names[true_labels_filtered[idx]]
        print(f"\n  Rank {rank} (actual category: {true_cat})")
        print(f"  Topic distribution: {dict(zip([f'T{i}' for i in range(4)], topic_dist.round(3)))}")
        print(f"  Text excerpt: {raw_docs_filtered[idx][:200].strip()}...")

In [ ]:
# ── VISUALIZE TOPIC DISTRIBUTIONS FOR SELECTED DOCUMENTS ────────────────────

# Pick one representative document per topic and show its full distribution
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
topic_colors = ['steelblue', 'coral', 'mediumseagreen', 'orchid']

for topic_id in range(4):
    # Get the document most dominated by this topic
    top_doc_idx = doc_topic_matrix[:, topic_id].argmax()
    topic_dist = doc_topic_matrix[top_doc_idx]

    axes[topic_id].bar(
        [f'T{i}' for i in range(4)],
        topic_dist,
        color=topic_colors,
        edgecolor='black',
        linewidth=0.5
    )
    axes[topic_id].set_title(f'Topic {topic_id}\n(most pure document)', fontsize=10)
    axes[topic_id].set_xlabel('Topic')
    axes[topic_id].set_ylabel('Probability')
    axes[topic_id].set_ylim(0, 1)
    axes[topic_id].axhline(0.5, color='red', linestyle='--', linewidth=0.8, alpha=0.7)

plt.suptitle('Document-Topic Distributions (Most Pure Document per Topic)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### How to Read This Chart: Document Topic Distribution

Each bar chart shows one document's **topic distribution** — the probability that each topic generated this document.

- **X-axis:** Topic labels (T0, T1, T2, T3)
- **Y-axis:** Probability (0 to 1). All 4 bars sum to 1.0.
- **The red dashed line at 0.5** is a visual reference point.

**What different shapes mean:**
- **One bar dominates (near 1.0):** A "pure" document that is almost entirely about one topic. These are the clearest, most useful training examples.
- **Two bars roughly equal (e.g., 0.5 and 0.4):** A mixed document that spans two topics — common in real-world text (e.g., a newsgroup post about a car accident that happened at a hospital).
- **All bars roughly equal (0.25 each):** The model is confused and can't assign this document to any topic clearly. Usually means the document is very short or contains mostly stopwords after preprocessing.

> **Practical insight:** If most of your documents look "flat" (equal probability across all topics), your topics are not well-separated. Try reducing the number of topics or improving preprocessing.

---

## Section 8 — BERTopic

### How BERTopic Works

> **BERTopic uses sentence-transformers to create semantic embeddings, then UMAP to reduce dimensions, then HDBSCAN to cluster, then c-TF-IDF to extract representative words per cluster.**

The full pipeline:

```
Documents
    ↓
SentenceTransformer (e.g., all-MiniLM-L6-v2)
    → Each document becomes a dense vector (384 dimensions)
    → Semantically similar documents have similar vectors
    ↓
UMAP (Uniform Manifold Approximation and Projection)
    → Reduces 384 dimensions → 5 dimensions
    → Preserves local structure (nearby points stay nearby)
    ↓
HDBSCAN (Hierarchical Density-Based Spatial Clustering)
    → Groups nearby points into clusters
    → Points that don't fit any cluster → Topic -1 (outliers)
    → Automatically determines the number of clusters!
    ↓
c-TF-IDF (class-based TF-IDF)
    → For each cluster, find the words that are most distinctive
    → Treat all documents in a cluster as one giant document
    → Compare against all other clusters
    ↓
Topics with representative words + documents
```

### Key Difference from LDA
LDA works with raw word counts — it has no idea that "automobile" and "car" mean the same thing. BERTopic uses BERT embeddings that encode semantic meaning, so synonyms and related concepts are naturally close together in vector space.

In [ ]:
# ── INSTALL AND IMPORT BERTOPIC ──────────────────────────────────────────────
# If not installed, uncomment:
# !pip install bertopic sentence-transformers

try:
    from bertopic import BERTopic
    from sentence_transformers import SentenceTransformer
    print("BERTopic and sentence-transformers imported successfully.")
    BERTOPIC_AVAILABLE = True
except ImportError:
    print("BERTopic is not installed.")
    print("Install with: pip install bertopic sentence-transformers")
    BERTOPIC_AVAILABLE = False

In [ ]:
# ── TRAIN BERTOPIC ───────────────────────────────────────────────────────────

if BERTOPIC_AVAILABLE:
    # Use the lightweight all-MiniLM-L6-v2 model (fast, 80MB, good quality)
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

    topic_model = BERTopic(
        embedding_model=embedding_model,
        language='english',
        calculate_probabilities=True,
        verbose=True,
        min_topic_size=15   # Minimum docs per topic (helps avoid micro-topics)
    )

    print("\nTraining BERTopic model...")
    print("(This will take a few minutes — BERT inference is slow on CPU)")

    # Use raw documents (not preprocessed) — BERTopic works better with natural language
    topics, probs = topic_model.fit_transform(raw_docs_filtered)

    print(f"\nDone! BERTopic discovered {len(set(topics)) - 1} topics (plus outliers).")
    print(f"Total documents: {len(topics)}")
    print(f"Outlier documents (Topic -1): {sum(1 for t in topics if t == -1)}")
else:
    print("BERTopic not available — skipping Sections 8-11.")
    print("Install with: pip install bertopic sentence-transformers")

In [ ]:
# ── PRINT BERTOPIC TOPICS ────────────────────────────────────────────────────

if BERTOPIC_AVAILABLE:
    topic_info = topic_model.get_topic_info()

    print("BERTOPIC DISCOVERED TOPICS")
    print("=" * 70)
    print(f"{'Topic':>6} | {'Count':>6} | Top Words")
    print("-" * 70)

    for _, row in topic_info.head(15).iterrows():
        topic_id = row['Topic']
        count = row['Count']
        if topic_id == -1:
            label = 'OUTLIERS'
        else:
            words = topic_model.get_topic(topic_id)
            label = ', '.join([w for w, _ in words[:6]])
        print(f"  {topic_id:>4} | {count:>6} | {label}")

    print(f"\n  ... (showing top 15 of {len(topic_info)} total entries including outliers)")

### How to Read This Output: BERTopic Topics

BERTopic prints a table of all discovered topics. Here's how to read each column:

- **Topic number:** Topics are numbered **0, 1, 2, 3, ...** in order of size. Topic 0 is the **largest topic** (most documents). Topic -1 is special — see below.
- **Count:** How many documents belong to this topic.
- **Top words:** The most representative words for this topic, ranked by c-TF-IDF score (how distinctive these words are compared to all other topics).

**Topic -1 — The Outlier Bucket:**
- HDBSCAN (the clustering algorithm inside BERTopic) assigns documents that don't fit any cluster to Topic -1.
- This is a **feature, not a bug** — it means the model is honest about uncertainty rather than forcing every document into a topic.
- If Topic -1 contains more than 20-30% of your documents, consider reducing `min_topic_size` or checking your data quality.

**Why BERTopic may find many more topics than expected:**
- Even with 4 ground-truth categories, BERTopic may discover 10, 20, or more topics.
- This is because BERT embeddings capture **subtopics within categories** — e.g., within `rec.autos` there might be sub-clusters for "buying advice", "engine problems", and "car models".
- Use `reduce_topics()` in the next section to merge them down to a manageable number.

---

## Section 9 — BERTopic Topic Reduction

BERTopic often discovers many more topics than you need — 20, 50, or even 100+ on a typical corpus. This is because the HDBSCAN clustering algorithm is sensitive to local density and will create a new cluster for every dense region in the embedding space.

**Topic reduction** merges similar topics together using cosine similarity between their c-TF-IDF vectors. The most similar pair of topics is merged first, then the next most similar pair, and so on — until you reach your target number of topics.

> **Important note:** Topic numbers change after reduction. Topic 0 in the reduced model is NOT the same topic as Topic 0 before reduction. Always inspect the top words after reducing.

In [ ]:
# ── TOPIC REDUCTION ──────────────────────────────────────────────────────────

if BERTOPIC_AVAILABLE:
    # Show topics BEFORE reduction
    n_before = len(topic_model.get_topic_info()) - 1  # subtract 1 for outlier topic -1
    print(f"Topics BEFORE reduction: {n_before}")
    print("\nTop 5 topics before reduction:")
    for _, row in topic_model.get_topic_info().head(6).iterrows():
        if row['Topic'] == -1:
            continue
        words = topic_model.get_topic(row['Topic'])
        print(f"  Topic {row['Topic']:>3} ({row['Count']:>4} docs): {', '.join([w for w, _ in words[:5]])}")

    # Reduce to 10 topics (or fewer if the model has fewer)
    target = min(10, n_before)
    print(f"\nReducing to {target} topics...")

    topic_model.reduce_topics(raw_docs_filtered, nr_topics=target)

    n_after = len(topic_model.get_topic_info()) - 1
    print(f"\nTopics AFTER reduction: {n_after}")
    print("\nTop 5 topics after reduction:")
    for _, row in topic_model.get_topic_info().head(6).iterrows():
        if row['Topic'] == -1:
            continue
        words = topic_model.get_topic(row['Topic'])
        print(f"  Topic {row['Topic']:>3} ({row['Count']:>4} docs): {', '.join([w for w, _ in words[:5]])}")

### How to Read This: Topic Reduction

The output shows topics before and after running `reduce_topics()`.

**Before reduction:** You may see many fine-grained subtopics — for example, within the "guns" category, BERTopic might separately identify clusters for "gun laws", "shooting incidents", and "weapon types".

**After reduction:** Similar topics are merged. The merged topics often have richer top-word lists because they now represent a broader theme.

**What to watch for:**
- Did the reduction merge topics from different ground-truth categories? (e.g., merged "cars" with "guns") — that would be bad. Check by looking at the top words.
- Did it correctly merge subtopics within the same category? (e.g., merged two car subtopics into one car topic) — that's good.

**When to reduce more aggressively:**
- If your end goal is a simple 4-category classification, reduce to 4.
- If you're exploring the data to understand themes, keep 8-15 topics for more nuance.

> **Key warning:** `reduce_topics()` modifies the model in-place. There is no "undo" button — always save the unreduced model first if you want to compare.

---

## Section 10 — BERTopic Visualizations

BERTopic has built-in visualization methods that generate interactive Plotly charts. We save them as HTML files for use outside the notebook.

In [ ]:
# ── BERTOPIC VISUALIZATIONS ──────────────────────────────────────────────────

if BERTOPIC_AVAILABLE:
    # 1. Topic distance map (intertopic distance map)
    print("Generating topic distance map...")
    try:
        fig_topics = topic_model.visualize_topics()
        fig_topics.write_html('bertopic_topic_map.html')
        print("  Saved: bertopic_topic_map.html")
    except Exception as e:
        print(f"  Could not generate topic map: {e}")

    # 2. Bar chart of top words per topic
    print("Generating topic word bar charts...")
    try:
        n_show = min(8, len(topic_model.get_topic_info()) - 1)
        fig_bars = topic_model.visualize_barchart(top_n_topics=n_show, n_words=8)
        fig_bars.write_html('bertopic_barchart.html')
        fig_bars.show()
        print("  Saved: bertopic_barchart.html")
    except Exception as e:
        print(f"  Could not generate bar chart: {e}")
else:
    print("BERTopic not available — skipping visualizations.")

### How to Read This Chart: BERTopic Topic Bars

Each sub-panel shows one topic. The horizontal bars show the **top words** for that topic.

- **X-axis (bar length) = c-TF-IDF score.** This is NOT a simple word frequency — it is a measure of how important this word is to THIS topic **relative to all other topics**.
  - A high c-TF-IDF score means: this word appears a lot in this topic AND rarely in other topics.
  - A word like "car" with high c-TF-IDF in Topic 0 means: documents in Topic 0 use "car" much more than documents in Topics 1, 2, 3 do.
- **Y-axis = word labels,** sorted from most to least distinctive.

**What to look for:**
- **Clean, coherent topics:** All words in the bar chart belong to the same domain (e.g., all car-related, all medicine-related). This means the topic is well-defined.
- **Mixed topics:** Words from different domains appear in the same bar chart. This usually means two topics got merged that shouldn't have been, or the underlying data overlaps heavily.
- **Very short bars:** Words with low c-TF-IDF scores — they appear in this topic but are also common in other topics. These are less informative for characterizing the topic.

> **Tip:** If the top words for a topic seem nonsensical (random strings, very short words, numbers), that topic may be an artifact of preprocessing issues — check that your text cleaning is working correctly.

---

## Section 11 — Zero-Shot Topic Classification with BERTopic

### What Is Zero-Shot Classification?

In standard topic modeling, you train a model and then interpret the topics it finds. In **zero-shot classification**, you provide topic labels in plain English and the model finds documents that match those labels — **without any retraining**.

BERTopic's `find_topics()` method works by:
1. Converting your query string to a BERT embedding (same embedding space as the document embeddings)
2. Computing cosine similarity between your query embedding and each topic's embedding
3. Returning the topics most similar to your query

This is powerful because you can say: *"Show me everything related to medical treatments"* and the model will find those documents even if it never saw that exact phrase during training.

In [ ]:
# ── ZERO-SHOT TOPIC SEARCH ───────────────────────────────────────────────────

if BERTOPIC_AVAILABLE:
    # Define our 4 semantic queries (matching the 4 ground-truth categories)
    queries = [
        "automobiles and cars",
        "medical science and health",
        "computer graphics and software",
        "politics and guns"
    ]

    print("ZERO-SHOT TOPIC SEARCH")
    print("=" * 70)

    for query in queries:
        print(f"\nQuery: '{query}'")
        print("-" * 50)

        try:
            similar_topics, similarities = topic_model.find_topics(query, top_n=3)
            for rank, (topic_id, sim) in enumerate(zip(similar_topics, similarities), 1):
                if topic_id == -1:
                    print(f"  Rank {rank}: Topic -1 (outliers) — similarity={sim:.4f}")
                    continue
                words = topic_model.get_topic(topic_id)
                word_str = ', '.join([w for w, _ in words[:6]])
                print(f"  Rank {rank}: Topic {topic_id} — similarity={sim:.4f}")
                print(f"           Top words: {word_str}")

                # Show one representative document from this topic
                topic_doc_indices = [i for i, t in enumerate(topic_model.topics_) if t == topic_id]
                if topic_doc_indices:
                    sample_idx = topic_doc_indices[0]
                    snippet = raw_docs_filtered[sample_idx][:150].replace('\n', ' ').strip()
                    print(f"           Sample doc: '{snippet}...'")
        except Exception as e:
            print(f"  Error during search: {e}")
else:
    print("BERTopic not available — skipping zero-shot classification.")

### How to Read This: Zero-Shot Classification

For each query, you see the top 3 matching topics along with their similarity scores.

**Similarity score:**
- This is **cosine similarity** between the query's BERT embedding and the topic's embedding (derived from its representative documents).
- Ranges from 0 (completely unrelated) to 1 (identical semantic meaning).
- In practice, scores above 0.5 indicate strong relevance; below 0.3 is a weak match.

**What to look for:**
- **High similarity between query and expected topic:** If "automobiles and cars" has high similarity to a topic whose top words are [car, engine, drive, vehicle], the model has successfully connected your query to the right cluster.
- **Cross-category matches:** If "politics and guns" matches a topic about military history with moderate similarity, that's meaningful — guns and military are semantically related in BERT's embedding space.
- **Query returning -1 (outliers) as a top match:** This can happen when the query is very general and the outlier cluster is large. Try making your query more specific.

**Practical applications:**
- Automatically categorize new documents without retraining
- Build a semantic search over your corpus ("find all documents about X")
- Validate that discovered topics align with domain expectations

---

## Section 12 — LDA vs BERTopic Comparison

Now that we've run both models on the same dataset, let's do a head-to-head comparison.

In [ ]:
# ── COMPUTE COHERENCE FOR BOTH MODELS ────────────────────────────────────────

print("Computing coherence scores for both models on the same data...\n")

# LDA coherence (already computed in Section 5 for n=4)
lda_topics_words = []
for topic in lda_model.components_:
    top_word_ids = topic.argsort()[-10:][::-1]
    top_words = [vocab[i] for i in top_word_ids if vocab[i] in dictionary.token2id]
    lda_topics_words.append(top_words)

lda_coherence_model = CoherenceModel(
    topics=lda_topics_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
lda_cv = lda_coherence_model.get_coherence()
print(f"LDA Coherence (c_v) with n=4 topics: {lda_cv:.4f}")

# BERTopic coherence (if available)
if BERTOPIC_AVAILABLE:
    bertopic_topics_words = []
    for topic_id in topic_model.get_topic_info()['Topic']:
        if topic_id == -1:
            continue
        words = topic_model.get_topic(topic_id)
        valid_words = [w for w, _ in words[:10] if w in dictionary.token2id]
        if valid_words:
            bertopic_topics_words.append(valid_words)

    if bertopic_topics_words:
        bertopic_coherence_model = CoherenceModel(
            topics=bertopic_topics_words,
            texts=tokenized_docs,
            dictionary=dictionary,
            coherence='c_v'
        )
        bertopic_cv = bertopic_coherence_model.get_coherence()
        print(f"BERTopic Coherence (c_v):             {bertopic_cv:.4f}")
    else:
        bertopic_cv = None
        print("BERTopic: Could not compute coherence (no valid topic words in gensim dictionary)")
else:
    bertopic_cv = None
    print("BERTopic not available — cannot compute BERTopic coherence.")

In [ ]:
# ── COMPARISON TABLE ─────────────────────────────────────────────────────────

bertopic_cv_str = f"{bertopic_cv:.4f}" if bertopic_cv else "N/A (not installed)"

comparison_data = {
    'Metric': [
        'Core Approach',
        'Text Representation',
        'Word Order',
        'Handles Short Texts',
        'Speed on 10K Docs',
        'Number of Topics',
        'Interpretability',
        'Handles Synonyms',
        'Requires Preprocessing',
        'Outlier Detection',
        'Coherence Score (c_v)',
    ],
    'LDA': [
        'Bayesian statistical model',
        'Bag-of-words (word counts)',
        'Completely ignored',
        'Poor (needs enough words)',
        'Very fast (~5 seconds)',
        'Must specify upfront',
        'Word probability distributions',
        'No (car ≠ automobile)',
        'Critical — must clean text',
        'No — forces all docs into topics',
        f'{lda_cv:.4f}',
    ],
    'BERTopic': [
        'Neural embeddings + density clustering',
        'Dense BERT embeddings (semantic)',
        'Captured by BERT embeddings',
        'Good (BERT handles short text well)',
        'Slow (~2-5 mins, GPU helps a lot)',
        'Auto-discovered by HDBSCAN',
        'Top words + representative documents',
        'Yes (car ≈ automobile in embedding space)',
        'Less critical (BERT is robust)',
        'Yes — Topic -1 for outliers',
        bertopic_cv_str,
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

### How to Read This Table: LDA vs BERTopic

This table summarizes when to choose each model. Here's the decision guide:

**Choose LDA when:**
- You have **long documents** (articles, papers, reports) with many words per document
- You have a **large corpus** (100K+ documents) and need speed
- You need a **simple, interpretable** model you can explain to stakeholders
- You already know roughly **how many topics** to expect
- You're running in a resource-constrained environment (no GPU)

**Choose BERTopic when:**
- You have **short texts** (tweets, product reviews, survey responses)
- You need **semantic understanding** (synonyms, paraphrases should map to the same topic)
- You **don't know how many topics** to expect and want the model to discover that
- You need to identify **outliers** (documents that don't fit any topic)
- You have GPU resources and quality matters more than speed

**About the coherence scores:**
- If BERTopic's coherence is higher, it's producing more semantically coherent topics (top words that naturally co-occur).
- But remember: BERTopic is also evaluated on **more topics** (after reduction), so the comparison isn't perfectly apples-to-apples.
- The best model depends on **your specific use case**, not just the coherence score.

---

## Section 13 — Summary and Key Takeaways

This notebook covered the complete topic modeling workflow from raw text to interpretable topics, using two fundamentally different approaches.

---

### When to Use Which Tool

| Scenario | Recommended Approach |
|----------|----------------------|
| Long documents, fast iteration, known topic count | **LDA** |
| Short texts (tweets, reviews), semantic understanding needed | **BERTopic** |
| Quick topic exploration, no neural stack | **LDA + TF-IDF** |
| Production system, need best quality, have GPU | **BERTopic** |
| Simple grouping without probabilistic topics | **TF-IDF + K-Means clustering** |

---

### Key Lesson 1: Preprocessing Matters More for LDA

LDA treats every unique string as a separate vocabulary item. If you don't lowercase, lemmatize, and remove stopwords:
- "Car", "car", "Cars", "car's" each count as different words
- "The", "is", "a" dominate every topic (they appear everywhere)
- Topics end up being mixtures of common words rather than distinct themes

BERTopic is more forgiving because BERT embeddings capture semantic meaning — "car" and "automobile" map to nearby points in embedding space even without normalization. But preprocessing still helps by removing noise.

---

### Key Lesson 2: Coherence Score Is Your Main Evaluation Metric

The **c_v coherence score** measures whether the top words of a topic naturally co-occur in documents. It correlates well with human judgments of topic quality.

- Use it to choose `n_topics` for LDA (the elbow/peak in the coherence curve)
- Use it to compare different preprocessing strategies
- Don't use **perplexity alone** — lower perplexity does not mean more interpretable topics

---

### Key Lesson 3: Topic Models Are Exploratory — There Is No "Correct" Answer

Topic modeling is an **unsupervised, exploratory** technique. Unlike classification, there is no ground truth telling you that your topics are right or wrong. Two valid models might produce different topics — and both can be useful.

Always validate topics by:
1. Reading the top words — do they make sense together?
2. Reading 5-10 example documents per topic — do they all belong?
3. Asking domain experts to review and name the topics

---

### Key Lesson 4: Common Mistakes to Avoid

| Mistake | Why It Hurts | Fix |
|---------|-------------|-----|
| Skipping preprocessing | Stopwords dominate topics; identical concepts treated as different words | Always lowercase, remove stopwords, lemmatize |
| Too many topics | Topics fragment into meaningless micro-clusters | Use coherence curve to find the elbow |
| Too few topics | Topics are vague, broad, overlap heavily | Same — use coherence curve |
| Not reading example documents | Top words look good but docs are actually from different domains | Always sanity-check by reading actual documents |
| Trusting perplexity alone | Model overfits statistically but topics are uninterpretable | Use coherence as primary metric |
| Comparing models on different data | LDA was preprocessed, BERTopic used raw text — coherence scores aren't directly comparable | Use identical preprocessing for fair comparison |

---

### What You Built in This Notebook

| Section | What You Did |
|---------|-------------|
| 1 | Set up environment and imported all libraries |
| 2 | Loaded 20 Newsgroups with 4 categories |
| 3 | Built a 5-step preprocessing pipeline; built CountVectorizer and TfidfVectorizer |
| 4 | Trained LDA with 4 topics; interpreted top words |
| 5 | Ran coherence/perplexity sweep over 7 topic counts; plotted model selection curves |
| 6 | Generated pyLDAvis interactive visualization (HTML export) |
| 7 | Explored document-topic assignments; visualized document mixtures |
| 8 | Trained BERTopic with BERT embeddings; printed discovered topics |
| 9 | Used `reduce_topics()` to merge BERTopic's fine-grained topics |
| 10 | Generated BERTopic visualizations (topic map + word bar charts) |
| 11 | Ran zero-shot topic search with 4 semantic queries |
| 12 | Compared LDA vs BERTopic across 11 dimensions + coherence scores |

---

### Next Steps

If you want to go deeper:
- **Dynamic Topic Modeling:** How topics change over time (BERTopic supports this with `topics_over_time()`)
- **Hierarchical Topic Modeling:** BERTopic's `hierarchical_topics()` shows how topics nest inside each other
- **Guided Topic Modeling:** Provide seed words to steer LDA toward known domains
- **Online LDA:** Update the model incrementally as new documents arrive without full retraining
- **Top2Vec:** A third approach similar to BERTopic but with a different clustering strategy

---

*End of notebook — Topic Modeling: LDA and BERTopic*  
*Author: Shivani Bokka*